# Занятие 10. Обучение одного нейрона методом градиентного спуска

Проследим, как loss подсказывает направление изменения веса. Затем обучим нейрон отвечать на четыре примера AND.

Запускайте ячейки сверху вниз. Для работы нужны NumPy и Matplotlib.

## 1. Измеряем наклон в точке
Посчитайте сами и предскажите, с какой стороны от `w = 0` значение *L* будет меньше. Замеряем его на расстоянии `h = 0.1` от точки.

In [ ]:
import numpy as np

def loss_w(w):
    return (w - 3)**2 + 1

def slope_at(f, w, h=0.1):
    return (f(w + h) - f(w - h)) / (2 * h)

w, h = 0, 0.1
print(f'Слева {loss_w(w-h):.2f}, справа {loss_w(w+h):.2f}')
print(f'Наклон около w=0: {slope_at(loss_w, w, h):.2f}')
new_w = w - 0.1 * slope_at(loss_w, w, h)
print(f'Новый w = {new_w:.2f}, loss {loss_w(w):.2f} → {loss_w(new_w):.2f}')

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(-1, 6, 120)
fig, ax = plt.subplots(figsize=(7, 3.3), layout='constrained')
ax.plot(x, loss_w(x), color='#245c8b')
for w_plot, label in [(0, 'Старт'), (new_w, 'Шаг 1'), (3, 'Дно')]:
    ax.scatter(w_plot, loss_w(w_plot), s=55)
    ax.annotate(label, (w_plot, loss_w(w_plot)), xytext=(9, 7), textcoords='offset points')
ax.set(xlabel='Параметр w', ylabel='L(w)', title='Начинаем с одного параметра')
ax.grid(alpha=0.2)
plt.show()

## 2. Меняем размер шага
Перед запуском сравните три значения learning rate. Какое даст медленный ход? При каком шаге точка может оказаться выше, чем раньше?

In [ ]:
def descend(start, lr, steps=12):
    w = start
    trace = [w]
    for _ in range(steps):
        w = w - lr * slope_at(loss_w, w)
        trace.append(w)
    return np.array(trace)

rates = [0.08, 0.4, 1.05]
for lr in rates:
    trace = descend(0, lr)
    print(f'lr={lr}: первые веса {trace[:4].round(2)}, итоговый loss={loss_w(trace[-1]):.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.3), layout='constrained')
for lr in rates:
    trace = descend(0, lr)
    ax.plot(range(len(trace)), loss_w(trace), marker='.', label=f'lr = {lr}')
ax.set(xlabel='Шаг', ylabel='L(w)', yscale='log')
ax.grid(alpha=0.2)
ax.legend()
plt.show()

## 3. Обучаем нейрон AND
Одна ручка превращается в три: `w1`, `w2` и `b`. Около текущей настройки измеряем три наклона: каждый раз пробуем сдвинуть только одну ручку. После этого обновляем все три вместе.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 0, 0, 1], dtype=float)
initial = np.array([0.1, -0.2, 0.1])

def predict(params):
    z = X @ params[:2] + params[2]
    return 1 / (1 + np.exp(-z))

def and_loss(params):
    return float(np.mean((predict(params) - y)**2))

def estimate_gradient(params, h=0.001):
    slopes = []
    for j in range(len(params)):
        shift = np.zeros_like(params)
        shift[j] = h
        slopes.append((and_loss(params + shift) - and_loss(params - shift)) / (2*h))
    return np.array(slopes)

def train_and(start, lr=0.8, steps=250):
    params = start.copy()
    history = [and_loss(params)]
    for _ in range(steps):
        params = params - lr * estimate_gradient(params)
        history.append(and_loss(params))
    return params, history

params, and_history = train_and(initial)
print('Первый градиент:', estimate_gradient(initial).round(3))
print(f'MSE: {and_history[0]:.4f} → {and_history[-1]:.4f}')
print('x | y | выход до | выход после | класс после')
for pair, label, before, after in zip(X, y, predict(initial), predict(params)):
    print(tuple(map(int, pair)), int(label), f'{before:.3f}', f'{after:.3f}', int(after > 0.5))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2), layout='constrained')
ax.plot(and_history, color='#245c8b')
ax.set(xlabel='Число обновлений', ylabel='MSE', title='Обучение нейрона AND')
ax.grid(alpha=0.2)
plt.show()

## Самостоятельно

### TODO 1. Сделайте один шаг из точки w = 5 при lr = 0.1. Выведите новую точку и обе ошибки. Перед запуском предскажите знак поправки.

In [ ]:
w = 5
lr = 0.1
# Здесь допишите вычисление new_w и вывод результатов.

### TODO 2. Повторите обучение AND всего за 25 обновлений. Сравните MSE и число правильных ответов с опытом на 250 обновлениях.

In [ ]:
steps = 25
# Вызовите train_and(initial, steps=...) и используйте predict.